In [ ]:
import pandas as pd
from tqdm import tqdm

WINDOW = 11
HALF = WINDOW // 2
BUFFER = 2

INPUT_FILE = "../data/proteasome_cleavage/virus_data_with_protein_sequences.csv"
OUTPUT_FILE = "../data/proteasome_cleavage/virus_cleavage_11mer_dataset.csv"

print("\nSTEP 1 — Loading dataset")
df = pd.read_csv(INPUT_FILE, low_memory=False)

print("Rows:", len(df))

# -------------------------
# STEP 2: Remove missing
# -------------------------

df = df.dropna(subset=[
    "protein_sequence",
    "Starting_Position",
    "Ending_Position"
])

print("Rows after filtering:", len(df))

# -------------------------
# STEP 3: Remove duplicate epitopes
# -------------------------

if "Clean Sequence" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset=["Clean Sequence","protein_id"])
    print("Removed duplicate epitopes:", before-len(df))

print("Rows after duplicate removal:", len(df))

# -------------------------
# STEP 4: Generate windows
# -------------------------

print("\nSTEP 4 — Generating cleavage windows")

data = []

for row in tqdm(df.itertuples(index=False), total=len(df)):

    seq = row.protein_sequence
    start = int(row.Starting_Position)
    end = int(row.Ending_Position)
    protein = row.protein_id

    cleavage = end - 1

    # POSITIVE WINDOW
    left = cleavage - HALF
    right = cleavage + HALF

    if left >= 0 and right < len(seq):

        window = seq[left:right+1]

        if len(window) == WINDOW:

            data.append({
                "protein_id": protein,
                "position": cleavage,
                "sequence": window,
                "label": 1
            })

    # NEGATIVE WINDOWS (internal residues)

    for pos in range(start+BUFFER, end-BUFFER):

        pos = pos - 1

        left = pos - HALF
        right = pos + HALF

        if left >= 0 and right < len(seq):

            window = seq[left:right+1]

            if len(window) == WINDOW:

                data.append({
                    "protein_id": protein,
                    "position": pos,
                    "sequence": window,
                    "label": 0
                })


# -------------------------
# STEP 5: Create dataset
# -------------------------

dataset = pd.DataFrame(data)

print("\nInitial dataset size:",len(dataset))
print(dataset.head())

# -------------------------
# STEP 6: Remove duplicates
# -------------------------

dataset = dataset.sort_values("label",ascending=False)

before=len(dataset)

dataset = dataset.drop_duplicates(subset=["sequence"])
print("Removed duplicate sequences:",before-len(dataset))

# -------------------------
# STEP 7: Remove negatives matching positives
# -------------------------

positive_set=set(dataset[dataset.label==1].sequence)

dataset = dataset[
    ~((dataset.label==0)&(dataset.sequence.isin(positive_set)))
]

print("\nFinal dataset size:",len(dataset))
print(dataset.label.value_counts())

# -------------------------
# STEP 8: Save dataset
# -------------------------

dataset.to_csv(OUTPUT_FILE,index=False)

print("\nDataset saved:",OUTPUT_FILE)


STEP 1 — Loading dataset
Rows: 22561
Rows after filtering: 22354
Removed duplicate epitopes: 0
Rows after duplicate removal: 22354

STEP 4 — Generating cleavage windows


100%|██████████| 22354/22354 [00:00<00:00, 57832.81it/s]



Initial dataset size: 120476
      protein_id  position     sequence  label
0  NCBI:P15059.2       303  ATCALVSDCAS      1
1  NCBI:P15059.2       297  YTEAAAATCAL      0
2  NCBI:P15059.2       298  TEAAAATCALV      0
3  NCBI:P15059.2       299  EAAAATCALVS      0
4  NCBI:P15059.2       300  AAAATCALVSD      0
Removed duplicate sequences: 47797

Final dataset size: 72679
label
0    53633
1    19046
Name: count, dtype: int64

Dataset saved: ../data/proteasome_cleavage/virus_cleavage_11mer_dataset.csv
